# 🫀 실험 10′ — 유도 ablation 확정판: **HYP를 10배로 보고, 동작점을 맞추고 잰다**

**MedKOS / `notebooks/exp10p_lead_ablation_cv.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 실험10이 남긴 세 가지 숙제

실험10은 목적을 달성했습니다 — `{I,II}`가 평균으로는 12유도의 **88.8%** 인데
HYP 환자에게는 **71.9%** 라는 걸 보였습니다. 그런데 주지표는 미결이었습니다:

```
교호작용({12} vs {II}) = +0.0646 [−0.0053, +0.1348]   ← 하한이 0을 0.005 차로 걸침
```

**세 가지를 고쳐야 합니다.**

### ① 표본을 늘리는 게 아니라 **평가 방식**을 바꿔야 한다

`QUICK=False`로 데이터를 3배로 늘려도 **HYP는 안 늘어납니다**:

```
HYP 535건  →  QUICK의 cap 1200에 애초에 안 걸렸다. 이미 전량 쓰고 있었다.
테스트 fold 10 = 전체의 10%  →  HYP 약 54건.  QUICK이든 Full이든 똑같이 54건.
```

흉부군 F1의 절반이 **54건짜리 클래스**에서 나오니 CI가 넓을 수밖에 없습니다.

> **해법: K겹 교차검증.** fold 하나만 테스트로 쓰지 말고 **모든 fold를 돌아가며**
> 테스트로 쓰면 HYP **535건 전부**가 평가에 들어갑니다 — **10배**입니다.
> (K는 테스트 커버리지가 아니라 *학습 표본 크기*를 정합니다. K≥2면 이미 전량 평가.)

### ② 라벨군을 **3분류**로 재정의한다

실험10에서 `{II}`→`{I,II}` 이득이 갈렸습니다:

| | `{II}`→`{I,II}` | 뜻 |
|---|---|---|
| **MI** | **+0.088** | 하벽 MI는 II·III·aVF에서 보인다 → **전두면을 열면 진단 가능** |
| **HYP** | +0.022 | 전압 기준이 흉부유도라 전두면으로는 안 열린다 |

`ailab-2026-0016` 2-bis가 이미 예고한 그대로입니다("하벽 STEMI: `{II}` B → `{I,II}` **A**").
**MI는 흉부군이 아니라 혼합군**이고, MI와 HYP를 한 군으로 묶은 게 신호를 흐렸습니다.

| 군 | 클래스 | 성격 |
|---|---|---|
| **전두면군** | NORM · CD · STTC | `{I,II}`로 열림 |
| **혼합군** | MI | 하벽=전두면 · 전벽/측벽=횡단면 |
| **횡단면군** | **HYP** | 흉부유도로만 열림 — **순수 신호** |

→ 주지표를 **횡단면군(HYP) 기준**으로 바꿉니다. 그게 가설이 말하는 바로 그 축입니다.

### ③ **동작점 정합을 사전등록 지표로 승격**한다

실험10에서 P3가 argmax로는 실패(Δ흉부 −0.001)했는데 동작점을 맞추니 **+0.043**으로
성립했습니다. **실험2·3에 이어 세 번째**이고, 이번엔 사전등록 채점을 뒤집는 크기였습니다.

> 세 번 걸렸으면 관행을 바꿔야 합니다. **주지표는 `{II}`의 오경보율에 맞춘 뒤 계산**하고,
> argmax는 참고로만 찍습니다.

---

## 사전등록

| # | 예측 | 근거 |
|---|---|---|
| **P1 (주가설)** | **교호작용 = ΔHYP − Δ전두면 > 0 유의** (동작점 정합 기준) | 흉부유도가 여는 것은 횡단면이고 HYP가 거기 산다 |
| **P2** | `{I,II}`에서 **ΔHYP ≈ 0** (≤ `{12}`가 HYP에서 얻는 것의 1/3) | `{I,II}`는 전두면 전체를 열 뿐 횡단면은 한 줄도 안 연다 |
| **P2b** | `{I,II}`에서 **ΔMI > ΔHYP** | 하벽 MI는 전두면에서 열린다(2-bis 표) |
| **P3′** | `{II,V1}`는 **ΔHYP를 못 올린다** (≤ `{12}`의 1/3) | 실험10에서 −0.086 관찰 → **예측을 뒤집어 재등록** |
| **P4** | agnostic의 `{II}` 손실 > `{12}` 손실 | 실험10에서 −0.178 vs −0.008 관찰 → 재검정 |

**P3′와 P4는 실험10의 관찰을 사전등록으로 승격한 것**입니다. 관찰을 다른 표본에서
예측으로 재검정하는 것이라, 맞으면 확증이고 틀리면 그 관찰이 우연이었다는 뜻입니다.

## 비용 — seed 앙상블을 버리고 테스트 표본을 산다

`5겹 × 5모델 × seed 1 = 25회 학습`. 사전등록했던 seed 3개를 **의도적으로 포기**합니다:

> 병목은 **점추정의 흔들림이 아니라 CI 폭**이었습니다. seed 앙상블은 점추정을
> 안정시키고, 교차검증은 CI를 좁힙니다. 지금 필요한 건 후자입니다.
> (5겹 평균 자체가 seed 잡음을 일부 상쇄하기도 합니다.)

**한계로 명시합니다.** 첫 학습 시간을 재서 전체 예상 시간을 출력하니, 너무 길면
`N_SEEDS`·`EPOCHS`를 줄이거나 `K_FOLD=3`으로 낮추세요.


In [ ]:
# CELL 1 — 설정
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

FS = 100
CLASSES  = ["NORM", "CD", "STTC", "MI", "HYP"]
# ★ 실험10의 2분류(전두면/흉부)를 3분류로 재정의한다.
#   MI는 하벽(전두면) + 전벽·측벽(횡단면)이 섞인 혼합군이라 한쪽에 넣으면 신호가 흐려진다.
G_FRONT = ["NORM", "CD", "STTC"]   # 전두면군 — {I,II}로 열림
G_MIXED = ["MI"]                   # 혼합군   — 하벽은 전두면, 전벽·측벽은 횡단면
G_TRANS = ["HYP"]                  # 횡단면군 — 흉부유도로만 열림 (★ 주지표)
LEADS12 = ["I", "II", "III", "AVR", "AVL", "AVF",
           "V1", "V2", "V3", "V4", "V5", "V6"]
CONFIGS = {"II": [1], "I+II": [0, 1], "II+V1": [1, 6], "12": list(range(12))}

# ★ 전량 사용. cap 없음 — 단 HYP는 535건이라 cap이 있으나 없으나 같았다(실험10의 교훈).
# ★★ 다수 클래스 언더샘플링 — 실험10 full 실행이 이것 없이 돌아 HYP가 붕괴했다.
#    QUICK의 cap=1200은 표본을 줄이는 지름길이 아니라 **우연한 클래스 균형기**였고,
#    그걸 없애니 [9069, 1708, 2400, 2532, 535]에서 모델이 NORM으로 쏠렸다
#    (HYP F1 0.441 → 0.088, NORM 0.530 → 0.780).
#    실험1′에서 검증된 처방을 그대로 쓴다: **오버샘플이 아니라 언더샘플**
#    (선행 트랙 N3/SupCon — 오버샘플은 inter-patient에서 해롭다).
BALANCE_RATIO = 4.0   # 각 클래스를 (최소 클래스 × 이 배수)까지만. 학습에만 적용.

K_FOLD  = 5        # 테스트 커버리지가 아니라 '학습 표본 크기'를 정한다. K≥2면 전량 평가.
N_SEEDS = 1        # ★ seed 앙상블을 버리고 교차검증을 산다 (아래 한계 참조)
EPOCHS  = 20
SEED0   = 20260801
BOOT    = 2000

CONFIG = dict(exp="exp10p_lead_ablation_cv", quest="ailab-2026-0015",
              parent_exp="exp10_lead",
              hypothesis="유도 이득은 라벨군마다 다르다 — 횡단면군(HYP) 기준 교호작용 > 0",
              primary_metric="interaction = ΔHYP − Δ전두면 (동작점 정합 기준)",
              prediction=("P1 교호작용>0 유의 · P2 {I,II}는 ΔHYP≈0 · P2b ΔMI>ΔHYP · "
                          "P3' {II,V1}는 ΔHYP 못 올림 · P4 agnostic {II}손실>{12}손실"),
              fix_1="K겹 교차검증 — HYP 535건 전량 평가(실험10은 54건)",
              fix_2="라벨군 3분류(전두면/혼합/횡단면) — MI는 혼합군",
              fix_3="동작점 정합을 사전등록 지표로 승격(실험2·3·10에서 세 번 걸림)",
              limitation="seed 앙상블 포기(N_SEEDS=1) — CI 폭이 병목이라 교차검증을 우선",
              dataset="ptb-xl 1.0.3 records100 전량", classes=CLASSES,
              groups={"frontal": G_FRONT, "mixed": G_MIXED, "transverse": G_TRANS},
              lead_configs={k: [LEADS12[i] for i in v] for k, v in CONFIGS.items()},
              balance_ratio=BALANCE_RATIO,
              fix_4=("학습셋 다수클래스 언더샘플링 — 실험10 full에서 HYP가 붕괴한 원인. "
                     "평가셋은 자연 유병률 그대로 둔다"),
              k_fold=K_FOLD, n_seeds=N_SEEDS, epochs=EPOCHS, fs=FS, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp10p_lead_cv", CONFIG, project=PROJECT)
run.log(f"교차검증 {K_FOLD}겹 · 유도구성 {len(CONFIGS)}종 · seed {N_SEEDS}")
run.log(f"총 학습 {(len(CONFIGS)+1) * K_FOLD * N_SEEDS}회 — 첫 학습 후 예상시간을 출력합니다")
run.log("라벨군: 전두면 " + str(G_FRONT) + " · 혼합 " + str(G_MIXED) +
        " · 횡단면 " + str(G_TRANS) + " ← 주지표")


In [ ]:
# CELL 2 — PTB-XL 확보 (실험10의 full 캐시를 그대로 재사용)
import wfdb, pandas as pd, subprocess, zipfile, shutil

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
ZIP = ("https://physionet.org/static/published-projects/ptb-xl/"
       "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip")

def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    return subprocess.run(["wget", "-q", "-O", dest, url]).returncode == 0 \
        and os.path.getsize(dest) > 0

for f in ("ptbxl_database.csv", "scp_statements.csv"):
    run.log(f"{'✅' if fetch(f'{BASE}/{f}', os.path.join(PTB, f)) else '❌'} {f}")

df = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)
run.log(f"레코드 {len(df):,} · 환자 {df.patient_id.nunique():,}")

# ★ 실험10을 QUICK=False로 돌렸다면 이 캐시가 이미 있다 — 다운로드도 재추출도 건너뛴다
CACHE = run.data("ptbxl_12lead_full.npz")
if os.path.exists(CACHE):
    run.log(f"✅ 실험10의 full 캐시 재사용: {CACHE}")
elif not os.path.isdir(os.path.join(PTB, "records100")):
    z = "/content/ptbxl.zip"
    run.log("전체 zip 내려받는 중(약 1.7GB, 5~15분)…")
    if not fetch(ZIP, z):
        raise RuntimeError("zip 다운로드 실패")
    with zipfile.ZipFile(z) as zf:
        zf.extractall("/content/_ptb")
    root = next(p for p, d, _ in os.walk("/content/_ptb") if "records100" in d)
    for name in os.listdir(root):
        src, dst = os.path.join(root, name), os.path.join(PTB, name)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    os.remove(z); shutil.rmtree("/content/_ptb", ignore_errors=True)
    run.log("✅ records100 준비 완료")
else:
    run.log("records100 이미 있음")


In [ ]:
# CELL 3 — 라벨 5종 + 3분류 라벨군
agg = scp[scp.diagnostic == 1].diagnostic_class.to_dict()
df["sc"] = df.scp_codes.apply(
    lambda s: sorted({agg[k] for k in ast.literal_eval(s) if k in agg}))
sub = df[df.sc.apply(lambda s: len(s) == 1 and s[0] in CLASSES)].copy()
sub["y"] = sub.sc.apply(lambda s: CLASSES.index(s[0]))

GRP = {}
for i, c in enumerate(CLASSES):
    GRP[i] = ("전두면" if c in G_FRONT else "혼합" if c in G_MIXED else "횡단면")
run.log(f"단일 superclass → **{len(sub):,}건** (cap 없음 · 전량)")
for i, c in enumerate(CLASSES):
    m = sub.y == i
    run.log(f"  {c:5s}({GRP[i]:3s}): {int(m.sum()):6,}건 · 환자 {sub[m].patient_id.nunique():,}명")

# ★ 실험10의 병목을 명시적으로 계산해 둔다 — 교차검증이 이걸 몇 배로 늘리는지
n_hyp = int((sub.y == CLASSES.index("HYP")).sum())
run.log(f"\n실험10에서는 테스트 fold 하나만 썼다 → HYP 약 {n_hyp // 10}건")
run.log(f"이번엔 {K_FOLD}겹 교차검증으로 **HYP {n_hyp}건 전량**이 평가에 들어간다 (약 10배)")
run.log(f"fold 분포: {sub.strat_fold.value_counts().sort_index().to_dict()}")


In [ ]:
# CELL 4 — 12유도 신호 (캐시 우선)
CACHE = run.data("ptbxl_12lead_full.npz")
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    X, Y, FOLD10, PID, EID = z["X"], z["y"], z["fold"], z["pid"], z["eid"]
    run.log(f"캐시 재사용: {X.shape} ({X.nbytes/1e6:.0f} MB, {X.dtype})")
    if len(X) != len(sub) or not np.array_equal(np.sort(EID), np.sort(sub.index.values)):
        raise RuntimeError("캐시가 지금 sub와 다르다 — CACHE를 지우고 다시 실행")
    sub = sub.loc[EID]
else:
    Xs, Ys, Fs_, Ps, Es, nofail = [], [], [], [], [], 0
    t0 = time.time()
    for k, (eid, row) in enumerate(sub.iterrows()):
        try:
            rec = wfdb.rdrecord(os.path.join(PTB, row.filename_lr))
        except Exception:
            nofail += 1; continue
        names = [n.strip().upper() for n in rec.sig_name]
        if any(n not in names for n in LEADS12):
            nofail += 1; continue
        s_ = np.nan_to_num(rec.p_signal)[:, [names.index(n) for n in LEADS12]].astype("float64")
        # 레코드 전체 스케일로 정규화 — 유도별로 나누면 유도 간 상대 진폭이 지워지고
        # 저전압·R파증가처럼 진폭 비교가 본질인 소견이 사라진다.
        q = np.percentile(s_, 75) - np.percentile(s_, 25)
        Xs.append(np.clip((s_ - np.median(s_)) / (q + 1e-6), -20, 20).astype("float16"))
        Ys.append(int(row.y)); Fs_.append(int(row.strat_fold))
        Ps.append(int(row.patient_id)); Es.append(int(eid))
        if (k + 1) % 2000 == 0:
            run.log(f"  {k+1}/{len(sub)} ({time.time()-t0:.0f}s)")
    X = np.array(Xs, "float16"); Y = np.array(Ys)
    FOLD10 = np.array(Fs_); PID = np.array(Ps); EID = np.array(Es)
    np.savez_compressed(CACHE, X=X, y=Y, fold=FOLD10, pid=PID, eid=EID)
    run.log(f"저장: {CACHE} ({X.nbytes/1e6:.0f} MB, 실패 {nofail}건)")
    sub = sub.loc[EID]

run.log(f"\nX{X.shape} · 클래스 {np.bincount(Y, minlength=5).tolist()} ({CLASSES})")
assert len(sub) == len(Y) and (sub.y.values == Y).all(), "sub와 배열 정렬 불일치"

# ★ 공식 strat_fold 1~10 → K겹으로 묶는다. 환자 계층화가 이미 되어 있으므로
#   fold 번호를 그대로 묶으면 환자 누수가 없다.
CV = (FOLD10 - 1) % K_FOLD
run.log(f"{K_FOLD}겹 배정 (공식 strat_fold를 묶음 — 환자 누수 없음)")
for k in range(K_FOLD):
    m = CV == k
    run.log(f"  겹 {k}: {m.sum():6,}건 · 클래스 {np.bincount(Y[m], minlength=5).tolist()}")


### CELL 5 — 물리 사전점검: `{I,II}` ≡ 사지유도 6개

P2·P2b가 여기 기댑니다. 성립하면 `{I,II}`에 III·aVR·aVL·aVF를 더하는 건 **증명 가능하게
무의미**하고, 그래도 HYP가 안 오르면 **"전두면에 그 정보가 없어서"** 가 확정됩니다.

실험10 실측은 최대 2.73%(aVR)였습니다. 독립 측정 픽스처에서는 40~54%가 나오므로
**선형종속 성립**입니다 — aVR만 큰 건 **I와 II를 더하는 유일한 식**이라 양자화 오차가
상쇄되지 않고 누적되기 때문입니다.


In [ ]:
# CELL 5 — 사지유도 선형종속 검증
IDX = {n: i for i, n in enumerate(LEADS12)}
def L(a, name): return a[:, :, IDX[name]]
chk = {"III  = II − I":    ("III", lambda a: L(a, "II") - L(a, "I")),
       "aVR  = −(I+II)/2": ("AVR", lambda a: -(L(a, "I") + L(a, "II")) / 2),
       "aVL  = I − II/2":  ("AVL", lambda a: L(a, "I") - L(a, "II") / 2),
       "aVF  = II − I/2":  ("AVF", lambda a: L(a, "II") - L(a, "I") / 2)}
smp = np.random.RandomState(SEED0).choice(len(X), min(400, len(X)), replace=False)
A = X[smp].astype("float32")
scale = float(np.percentile(np.abs(A), 99))
run.log(f"표본 {len(smp)}건 · 99퍼센타일 진폭 {scale:.3f}")
geo = {}
for name, (tgt, fn) in chk.items():
    err = np.abs(L(A, tgt) - fn(A))
    rel = float(err.mean() / (scale + 1e-9))
    geo[name] = rel
    # 문턱 근거: 독립 측정이면 40~54%가 나온다. 100Hz 다운샘플·양자화 오차는 수 % 수준.
    ok = "✅" if rel < 0.05 else ("⚠️" if rel < 0.15 else "❌")
    run.log(f"  {ok} {name:18s} 진폭의 {rel*100:5.2f}%")
worst = max(geo.values())
if worst < 0.05:
    run.log(f"→ **선형종속 확인**(최대 {worst*100:.1f}%). `{{I,II}}` ≡ 사지유도 6개.")
    run.log("   P2가 참이면 '유도를 덜 줘서'가 아니라 **'전두면에 정보가 없어서'** 다.")
else:
    run.log(f"→ ⚠️ 최대 {worst*100:.1f}% — 양자화로 보기엔 크다. P2 해석에 함께 적을 것.")
run.save_json("precheck_lead_geometry", {"rel_errors": geo, "worst_rel": worst})


In [ ]:
# CELL 6 — 교차검증 학습 (겹 × 구성 × seed, 체크포인트 있음)
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score, confusion_matrix

NC = len(CLASSES)
def backbone(inp):
    x = inp
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    return layers.GlobalAveragePooling1D()(x)

def build(n_ch, seed, agnostic=False):
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], n_ch)); ins = [si]
    h = layers.Dense(64, activation="relu")(backbone(si))
    if agnostic:
        mi = layers.Input((12,)); ins.append(mi)
        h = layers.Concatenate()([h, layers.Dense(16, activation="relu")(mi)])
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(ins, layers.Dense(NC, activation="softmax")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy")
    return m

def mask_of(cfg):
    m = np.zeros(12, "float32"); m[CONFIGS[cfg]] = 1.0
    return m

class RLMSeq(tf.keras.utils.Sequence):
    """배치마다 유도 구성을 무작위로 골라 나머지를 0으로 (Random Lead Masking)."""
    def __init__(self, idx, bs, seed, sw, shuffle=True):
        self.idx, self.bs, self.shuffle, self.sw = idx, bs, shuffle, sw
        self.names = list(CONFIGS); self.rs = np.random.RandomState(seed)
        self.order = idx.copy(); self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.idx) / self.bs))
    def on_epoch_end(self):
        if self.shuffle: self.rs.shuffle(self.order)
    def __getitem__(self, i):
        b = self.order[i * self.bs:(i + 1) * self.bs]
        cfg = self.names[self.rs.randint(len(self.names))] if self.shuffle \
              else self.names[i % len(self.names)]
        m = mask_of(cfg)
        return (X[b].astype("float32") * m, np.repeat(m[None], len(b), 0)), Y[b], self.sw[b]

# out-of-fold 확률을 담을 그릇
OOF = {f"fixed_{c}": np.zeros((len(Y), NC)) for c in CONFIGS}
OOF.update({f"agno_{c}": np.zeros((len(Y), NC)) for c in CONFIGS})

def auto_weights(y, beta=0.9999):
    w = {c: (1 - beta) / (1 - beta ** max((y == c).sum(), 1)) for c in range(NC)}
    return {c: float(v / w[0]) for c, v in w.items()}

t_start, n_done, n_total = time.time(), 0, (len(CONFIGS) + 1) * K_FOLD * N_SEEDS
for k in range(K_FOLD):
    te = CV == k
    rest = np.where(~te)[0]
    rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
    n_val = max(int(len(rest) * 0.12), 200)
    va_i, tr_i_raw = rest[:n_val], rest[n_val:]
    te_i = np.where(te)[0]

    # ★ 학습셋만 균형을 맞춘다. 평가셋(te_i)은 자연 유병률 그대로 — 배포 조건이 그렇다.
    def balance(idx, ratio, seed):
        rs = np.random.RandomState(seed)
        cnt = np.bincount(Y[idx], minlength=NC)
        cap = int(cnt[cnt > 0].min() * ratio)
        out = []
        for c in range(NC):
            ci = idx[Y[idx] == c]
            out.append(rs.choice(ci, cap, replace=False) if len(ci) > cap else ci)
        return np.sort(np.concatenate(out))
    tr_i = balance(tr_i_raw, BALANCE_RATIO, SEED0 + k)
    run.log(f"  학습 균형: {np.bincount(Y[tr_i_raw], minlength=NC).tolist()} → "
            f"{np.bincount(Y[tr_i], minlength=NC).tolist()} "
            f"({len(tr_i_raw):,} → {len(tr_i):,}건)")

    CW = auto_weights(Y[tr_i]); SWALL = np.zeros(len(Y), "float32")
    SWALL[tr_i] = [CW[int(c)] for c in Y[tr_i]]
    run.log(f"\n── 겹 {k}: 학습 {len(tr_i):,} / 검증 {len(va_i):,} / 테스트 {len(te_i):,}")

    for cfg, leads in CONFIGS.items():
        arm = f"fixed_{cfg}_f{k}"
        c_ = run.load_arm(arm)
        if c_ is None:
            acc = np.zeros((len(te_i), NC))
            for s in range(N_SEEDS):
                m = build(len(leads), SEED0 + 100 * k + s)
                h = m.fit(X[tr_i][:, :, leads].astype("float32"), Y[tr_i],
                          validation_data=(X[va_i][:, :, leads].astype("float32"), Y[va_i]),
                          epochs=EPOCHS, batch_size=128,
                          sample_weight=SWALL[tr_i], verbose=0)
                acc += m.predict(X[te_i][:, :, leads].astype("float32"),
                                 batch_size=512, verbose=0)
                tf.keras.backend.clear_session(); n_done += 1
                if n_done == 1:
                    per = time.time() - t_start
                    run.log(f"  ⏱ 첫 학습 {per:.0f}s → 전체 {n_total}회 예상 "
                            f"**{per*n_total/60:.0f}분**. 길면 K_FOLD·EPOCHS를 낮추세요.")
            c_ = acc / N_SEEDS
            run.save_arm(arm, c_)
        else:
            n_done += N_SEEDS
        OOF[f"fixed_{cfg}"][te_i] = c_
        run.log(f"  fixed_{cfg:<6s} 완료 ({n_done}/{n_total} · {time.time()-t_start:.0f}s)")

    need = [c for c in CONFIGS if run.load_arm(f"agno_{c}_f{k}") is None]
    if need:
        accs = {c: np.zeros((len(te_i), NC)) for c in CONFIGS}
        for s in range(N_SEEDS):
            m = build(12, SEED0 + 100 * k + 50 + s, agnostic=True)
            m.fit(RLMSeq(tr_i, 128, SEED0 + k + s, SWALL),
                  validation_data=RLMSeq(va_i, 256, SEED0, SWALL, shuffle=False),
                  epochs=EPOCHS, verbose=0)
            for cfg in CONFIGS:
                mk = mask_of(cfg)
                accs[cfg] += m.predict([X[te_i].astype("float32") * mk,
                                        np.repeat(mk[None], len(te_i), 0)],
                                       batch_size=512, verbose=0)
            if k == 0 and s == 0:
                run.save_model(m, "agnostic")   # 실험 11·12가 이어받는다
            tf.keras.backend.clear_session(); n_done += 1
        for cfg in CONFIGS:
            run.save_arm(f"agno_{cfg}_f{k}", accs[cfg] / N_SEEDS)
    else:
        n_done += N_SEEDS
    for cfg in CONFIGS:
        OOF[f"agno_{cfg}"][te_i] = run.load_arm(f"agno_{cfg}_f{k}")
    run.log(f"  agnostic 완료 ({n_done}/{n_total} · {time.time()-t_start:.0f}s)")

run.log(f"\n총 {time.time()-t_start:.0f}s · out-of-fold 예측 {len(Y):,}건 완성")


In [ ]:
# CELL 7 — 평가: 동작점 정합이 **주지표**, argmax는 참고
GI = {"전두면": [CLASSES.index(c) for c in G_FRONT],
      "혼합":   [CLASSES.index(c) for c in G_MIXED],
      "횡단면": [CLASSES.index(c) for c in G_TRANS]}
norm_m = (Y == 0)

def gf1(pred, grp):
    f = f1_score(Y, pred, average=None, labels=range(NC), zero_division=0)
    return float(np.mean([f[i] for i in grp]))

# ── 동작점 정합: {II}의 오경보율에 모든 구성을 맞춘다.
#   α는 전체 데이터에서 한 번만 정하고 부트스트랩 안에서는 고정한다
#   (동작점은 '고정된 결정규칙'이지 재표본마다 다시 튜닝할 대상이 아니다).
def alpha_for(prob, target):
    lo, hi = 0.02, 50.0
    for _ in range(40):
        mid = (lo * hi) ** 0.5
        p = prob.copy(); p[:, 0] *= mid
        if float((p.argmax(1)[norm_m] != 0).mean()) > target: lo = mid
        else: hi = mid
    return hi

ref_fa = float((OOF["fixed_II"].argmax(1)[norm_m] != 0).mean())
run.log(f"기준 오경보율 = 유도고정 {{II}}의 {ref_fa:.3f}")
if not (0.02 < ref_fa < 0.95):
    run.log(f"  ⛔ 기준 오경보율이 극단({ref_fa:.3f})이다. 동작점 정합이 의미를 잃으므로")
    run.log("     α를 모두 1로 두고 argmax로 진행한다 — 주지표 승격은 이번엔 적용 불가.")
MATCH_OK = bool(0.02 < ref_fa < 0.95)
ALPHA, PRED_M, PRED_A = {}, {}, {}
for arm, pr in OOF.items():
    a = alpha_for(pr, ref_fa) if MATCH_OK else 1.0; ALPHA[arm] = a
    p = pr.copy(); p[:, 0] *= a
    PRED_M[arm] = p.argmax(1)          # 동작점 정합 (주지표)
    PRED_A[arm] = pr.argmax(1)         # argmax (참고)

run.log("\n" + "=" * 96)
run.log("【표 1】 전체 평균만 볼 때 — 축소유도 챌린지들의 보고 방식")
run.log("=" * 96)
base = f1_score(Y, PRED_M["fixed_12"], average="macro", zero_division=0)
for cfg in CONFIGS:
    mf = f1_score(Y, PRED_M[f"fixed_{cfg}"], average="macro", zero_division=0)
    run.log(f"  {cfg:<7}{mf:>10.4f}   12유도 대비 {mf/base*100:>5.1f}%")
run.log("  ↑ 이 표만 보면 '적은 유도로도 대부분 나온다'로 읽힌다.")

run.log("\n" + "=" * 96)
run.log("【표 2】 라벨군 3분류 — 평균이 가린 것 (동작점 정합 기준)")
run.log("=" * 96)
run.log(f"  {'구성':<7}{'전두면':>9}{'혼합(MI)':>10}{'횡단면(HYP)':>13}   " +
        "".join(f"{c:>8}" for c in CLASSES))
grp_f1 = {}
for cfg in CONFIGS:
    p = PRED_M[f"fixed_{cfg}"]
    grp_f1[cfg] = {g: gf1(p, idx) for g, idx in GI.items()}
    per = f1_score(Y, p, average=None, labels=range(NC), zero_division=0)
    run.log(f"  {cfg:<7}{grp_f1[cfg]['전두면']:>9.4f}{grp_f1[cfg]['혼합']:>10.4f}"
            f"{grp_f1[cfg]['횡단면']:>13.4f}   " +
            "".join(f"{per[i]:>8.3f}" for i in range(NC)))
for g in GI:
    v = grp_f1["I+II"][g] / (grp_f1["12"][g] + 1e-9) * 100
    run.log(f"  {{I,II}}는 {g}군에서 12유도의 {v:.1f}%")

# ★ 학습 붕괴 가드: {12} ⊃ {II} 이므로 정보가 늘었는데 클래스 F1이 떨어지면
#   그건 발견이 아니라 최적화 실패다(물리적으로 불가능). 실험10 full에서 HYP가
#   0.165 → 0.088로 떨어졌고, 그걸 '예측 반전'으로 읽을 뻔했다.
f_ii = f1_score(Y, PRED_M["fixed_II"], average=None, labels=range(NC), zero_division=0)
f_12 = f1_score(Y, PRED_M["fixed_12"], average=None, labels=range(NC), zero_division=0)
degraded = [(CLASSES[i], float(f_ii[i]), float(f_12[i]))
            for i in range(NC) if f_12[i] < f_ii[i] - 0.02]
if degraded:
    run.log("\n" + "⛔" * 30)
    run.log("학습 붕괴 의심 — {12}는 {II}를 완전히 포함하는데 F1이 떨어진 클래스가 있다:")
    for c, a, b in degraded:
        run.log(f"   {c}: {{II}} {a:.3f} → {{12}} {b:.3f}  (Δ{b-a:+.3f})")
    run.log("   정보가 늘었는데 성능이 떨어지는 것은 물리적으로 불가능하다 = 최적화 실패.")
    run.log("   BALANCE_RATIO를 낮추거나 EPOCHS를 늘려야 한다. 아래 판정은 무효로 볼 것.")
    run.log("⛔" * 30)

# ── 주지표: 교호작용 (부트스트랩, α 고정)
def boot_inter(hi_arm, lo_arm, grp, B=BOOT, seed=SEED0):
    rs = np.random.RandomState(seed); n = len(Y); out = []
    ph, pl = PRED_M[hi_arm], PRED_M[lo_arm]
    for _ in range(B):
        i = rs.randint(0, n, n); y = Y[i]
        def g(p, gg):
            f = f1_score(y, p[i], average=None, labels=range(NC), zero_division=0)
            return np.mean([f[j] for j in gg])
        out.append((g(ph, grp) - g(pl, grp)) -
                   (g(ph, GI["전두면"]) - g(pl, GI["전두면"])))
    out = np.array(out)
    return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

run.log("\n" + "=" * 96)
run.log("【주지표】 교호작용 = Δ(해당군) − Δ(전두면군) · 동작점 정합 기준")
run.log("=" * 96)
inter = {}
for cfg in ("I+II", "II+V1", "12"):
    for gname in ("횡단면", "혼합"):
        d_g = grp_f1[cfg][gname] - grp_f1["II"][gname]
        d_f = grp_f1[cfg]["전두면"] - grp_f1["II"]["전두면"]
        m_, l_, u_ = boot_inter(f"fixed_{cfg}", "fixed_II", GI[gname])
        sig = bool(l_ > 0 or u_ < 0)
        inter[f"{cfg}|{gname}"] = {"delta_group": d_g, "delta_frontal": d_f,
                                   "interaction": m_, "ci": [l_, u_], "significant": sig}
        star = " ★" if sig else ""
        run.log(f"  {cfg:>6s} vs II │ {gname} │ Δ군 {d_g:+.4f} · Δ전두면 {d_f:+.4f} │ "
                f"교호작용 {m_:+.4f} [{l_:+.4f}, {u_:+.4f}]{star}")

# ── 사전등록 채점
run.log("\n" + "=" * 96)
run.log("【사전등록 채점】")
run.log("=" * 96)
V = {}
i12 = inter["12|횡단면"]
V["P1"] = bool(i12["significant"] and i12["interaction"] > 0)
run.log(f"  P1 교호작용(횡단면, {{12}} vs {{II}}) > 0 유의 → "
        f"{'✅' if V['P1'] else '❌'} ({i12['interaction']:+.4f} "
        f"[{i12['ci'][0]:+.4f}, {i12['ci'][1]:+.4f}])")

d_hyp_i2 = inter["I+II|횡단면"]["delta_group"]
d_fr_i2  = inter["I+II|횡단면"]["delta_frontal"]
d_hyp_12 = inter["12|횡단면"]["delta_group"]
# ★ 기준을 'Δ전두면의 절반'이 아니라 '{12}가 HYP에서 얻는 것의 1/3'로 잡는다.
#   Δ전두면이 음수/0근처면 원래 식은 무의미해진다(픽스처에서 확인). P3′와도 형태가 같아진다.
V["P2"] = bool(d_hyp_i2 <= d_hyp_12 / 3)
run.log(f"  P2 {{I,II}}는 ΔHYP를 거의 못 올림 (≤ {{12}}의 1/3) → {'✅' if V['P2'] else '❌'} "
        f"(ΔHYP {d_hyp_i2:+.4f} vs {{12}} {d_hyp_12:+.4f} · 참고 Δ전두면 {d_fr_i2:+.4f})")

d_mi_i2 = inter["I+II|혼합"]["delta_group"]
V["P2b"] = bool(d_mi_i2 > d_hyp_i2)
run.log(f"  P2b {{I,II}}에서 ΔMI > ΔHYP → {'✅' if V['P2b'] else '❌'} "
        f"(ΔMI {d_mi_i2:+.4f} vs ΔHYP {d_hyp_i2:+.4f})  ← 하벽 MI는 전두면에서 열린다")

d_hyp_v1 = inter["II+V1|횡단면"]["delta_group"]
V["P3p"] = bool(d_hyp_v1 <= d_hyp_12 / 3)
run.log(f"  P3′ {{II,V1}}는 ΔHYP를 못 올림 (≤ {{12}}의 1/3) → {'✅' if V['P3p'] else '❌'} "
        f"(ΔHYP {d_hyp_v1:+.4f} vs {{12}} {d_hyp_12:+.4f})")

ag = {c: f1_score(Y, PRED_M[f"agno_{c}"], average="macro", zero_division=0) for c in CONFIGS}
fx = {c: f1_score(Y, PRED_M[f"fixed_{c}"], average="macro", zero_division=0) for c in CONFIGS}
loss = {c: ag[c] - fx[c] for c in CONFIGS}
V["P4"] = bool(loss["II"] < loss["12"])
run.log(f"  P4 agnostic의 {{II}} 손실 > {{12}} 손실 → {'✅' if V['P4'] else '❌'} "
        f"({loss['II']:+.4f} vs {loss['12']:+.4f})")

run.log("\n" + "=" * 96)
run.log("【실험9 재검정】 lead-agnostic 1개 vs 유도고정 4개 (동작점 정합)")
run.log("=" * 96)
run.log(f"  {'구성':<7}{'고정':>9}{'agnostic':>11}{'차이':>9}"
        f"{'고정 횡단면':>13}{'agno 횡단면':>13}")
ag_cmp = {}
for cfg in CONFIGS:
    gh = gf1(PRED_M[f"agno_{cfg}"], GI["횡단면"])
    fh = gf1(PRED_M[f"fixed_{cfg}"], GI["횡단면"])
    ag_cmp[cfg] = {"fixed": fx[cfg], "agnostic": ag[cfg], "loss": loss[cfg],
                   "fixed_trans": fh, "agno_trans": gh}
    run.log(f"  {cfg:<7}{fx[cfg]:>9.4f}{ag[cfg]:>11.4f}{loss[cfg]:>+9.4f}"
            f"{fh:>13.4f}{gh:>13.4f}")

# ── argmax 대조 (실험10과 비교하려고)
run.log("\n[참고] argmax 기준 — 실험10과 같은 계산법")
for cfg in CONFIGS:
    p = PRED_A[f"fixed_{cfg}"]
    run.log(f"  {cfg:<7} 전체 {f1_score(Y, p, average='macro', zero_division=0):.4f} · "
            f"전두면 {gf1(p, GI['전두면']):.4f} · 혼합 {gf1(p, GI['혼합']):.4f} · "
            f"횡단면 {gf1(p, GI['횡단면']):.4f} · α={ALPHA[f'fixed_{cfg}']:.2f}")

hits = sum(V.values())
if degraded:
    verdict = (f"판정 무효 — 학습 붕괴({[c for c,_,_ in degraded]}에서 {{12}}가 {{II}}보다 "
               "나쁘다). 클래스 균형·에폭을 고쳐 재실행할 것")
elif V["P1"] and V["P2"]:
    verdict = ("확증 — 유도의 이득은 라벨군마다 다르다. 전두면을 아무리 채워도 횡단면군은 "
               "안 열린다. **유도 '개수'가 아니라 '평면'이 결정한다**")
elif V["P1"]:
    verdict = "부분 확증 — 교호작용은 유의하나 P2가 깨졌다. 전두면/횡단면 이분법이 덜 깨끗하다"
elif i12["interaction"] > 0:
    verdict = ("미결 — 방향은 맞으나 CI가 0을 걸친다. 교차검증으로 HYP를 10배 늘렸는데도 "
               "부족하다면 표본이 아니라 효과 크기의 문제일 수 있다")
else:
    verdict = "예측 반전 — 유도 추가가 전두면군을 더 올렸다. 라벨·정규화 재점검 필요"
run.log("\n" + "=" * 96)
run.log(f"▶ 사전등록 {hits}/5 적중")
run.log(f"▶ {verdict}")
run.log("=" * 96)

run.save_json("evaluation", {
    "group_f1_matched": grp_f1, "interaction": inter, "predictions": V,
    "alpha": ALPHA, "ref_false_alarm": ref_fa, "agnostic_vs_fixed": ag_cmp,
    "confusion_matched": {a: confusion_matrix(Y, p, labels=range(NC)).tolist()
                          for a, p in PRED_M.items()},
    "collapse_check": {"degraded_classes": degraded}, "verdict": verdict})

result = {"week": 2, "exp_id": "exp10p_lead_cv", "quest": "ailab-2026-0015",
          "task": "유도 ablation 확정판 — 교차검증 + 라벨군 3분류 + 동작점 정합",
          "split": "inter", "metric": "interaction_transverse_minus_frontal_matched",
          "value": round(i12["interaction"], 4),
          "passed": bool(V["P1"] and V["P2"]), "date": time.strftime("%Y-%m-%d"),
          "n_records": int(len(Y)), "k_fold": K_FOLD, "n_seeds": N_SEEDS,
          "n_hyp_evaluated": int((Y == CLASSES.index("HYP")).sum()),
          "group_f1_matched": grp_f1, "interaction": inter, "predictions": V,
          "agnostic_vs_fixed": ag_cmp, "degraded_classes": degraded, "verdict": verdict,
          "summary": (f"교호작용(횡단면,{{12}}vs{{II}}) {i12['interaction']:+.4f} "
                      f"[{i12['ci'][0]:+.4f},{i12['ci'][1]:+.4f}] · "
                      f"HYP {int((Y==CLASSES.index('HYP')).sum())}건 전량평가 · "
                      f"예측 {hits}/5 → {verdict.split(' —')[0]}")}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp10p_lead_ablation_cv.ipynb \\
      --quest ailab-2026-0015 --step "exp10p-lead-ablation-cv" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")


---

## 결과 읽는 법

| P1 | P2 | 뜻 | 다음 |
|---|---|---|---|
| **유의 +** | ✅ | **확증.** 유도 개수가 아니라 **평면**이 결정한다. 축소유도 논문의 "95% 성능"이 라벨군 평균이 만든 착시임을 실증 | 배포 정책 확정: 웨어러블은 전두면군 담당, 횡단면군은 **abstain**. `ailab-2026-0016` 3-tier를 모델 출력에 직결 |
| 유의 + | ❌ | `{I,II}`도 횡단면군을 올렸다 | 전두면/횡단면 이분법이 덜 깨끗하다. HYP의 사지유도 전압 기준(저전압 등)이 원인일 수 있다 |
| 비유의 | — | **HYP를 10배로 늘려도 안 잡혔다** | 표본이 아니라 효과 크기의 문제. HYP 자체가 어려운 클래스(F1 0.3~0.4)라 천장이 낮다 → 라벨을 세분(LVH/RVH)하거나 다른 데이터셋 추가 |

## 이 실험이 남기는 것

1. **HYP 535건 전량에 대한 out-of-fold 예측** — 실험10의 54건 대비 10배
2. **lead-agnostic 가중치**(`arms/agnostic/weights.keras`) — 실험 9b·11·12가 이어받는다
3. **라벨군 3분류 표** — 배포 정책(어느 유도에서 어느 군을 포기하나)의 근거
4. **동작점 정합을 주지표로 쓴 첫 실험** — 실험2·3·10에서 세 번 걸린 뒤의 관행 변경

## 한계 (헤드라인과 함께 보고)

- **seed 앙상블을 포기했다**(`N_SEEDS=1`). CI 폭이 병목이라 교차검증을 우선했다.
  5겹 평균이 seed 잡음을 일부 상쇄하지만 사전등록했던 3 seed보다 약하다.
- **HYP는 여전히 535건**이다. 교차검증이 평가 표본을 10배로 늘렸을 뿐 **학습** 표본은
  그대로다. HYP F1의 천장 자체가 낮을 수 있다.
- `{II,V1}`의 V1은 **표준 흉부전극 위치**다. 임상 텔레메트리의 MCL1은 위치가 다르므로
  여기 수치는 **상한**이다 → 실험 11(전극위치 augmentation)에서 갭을 따로 잰다.
